# Imports

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
%load_ext autoreload
%autoreload 2

# import NN_Classes.NN_Core as nn_core
# import NN_Classes.NN_Modules as nn_mods
# import NN_Classes.NN_Operations as nn_ops
# import NN_Classes.NN_Optimization as nn_opt
# import NN_Classes.NN_Data_Tracking as nn_data
# import NN_Classes.NN_Helpers as nn_helpers
import NN_Classes as nn

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Digit Classification

## Digit Data

In [7]:
processed_kaggle_data_path = "/Users/ronitmathur/Desktop/ML/Feed-Forward_NN/Data/Processed_Data/Kaggle_Digits/"
digit_X_train = np.load(f"{processed_kaggle_data_path}/X_train.npy")
digit_Y_train = np.load(f"{processed_kaggle_data_path}/Y_train.npy")
digit_X_validation = np.load(f"{processed_kaggle_data_path}/X_validation.npy")
digit_Y_validation = np.load(f"{processed_kaggle_data_path}/Y_validation.npy")
digit_X_test = np.load(f"{processed_kaggle_data_path}/X_test.npy")

# converting into desired shape
digit_X_train = digit_X_train.T
digit_Y_train = digit_Y_train.T
digit_X_validation = digit_X_validation.T
digit_Y_validation = digit_Y_validation.T
digit_X_test = digit_X_test.T

digit_X_train.shape

(784, 33600)

## Running on Digits

### Module-Test

In [11]:
module_test_model = nn.Sequential(
    nn.Linear(784, 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 10)
)
module_test_optimizer = nn.SGD(module_test_model, learning_rate=0.85)


# module_test_model = nn_mods.Sequential(
#     nn_mods.Linear(784, 50),
#     nn_mods.ReLU(),
#     nn_mods.Linear(50, 25),
#     nn_mods.ReLU(),
#     nn_mods.Linear(25, 10)

#     # nn_mods.Linear(784, 128),
#     # nn_mods.ReLU(),
#     # nn_mods.Linear(128, 64),
#     # nn_mods.ReLU(),
#     # nn_mods.Linear(64, 10)
# )
# # module_test_optimizer = nn_opt.SGD(module_test_model, learning_rate=0.01)
# module_test_optimizer = nn_opt.SGD(module_test_model, learning_rate=1)

In [12]:
epochs = 200

for epoch in range(1, epochs + 1):

    # data
    x_train_data = nn.Tensor(digit_X_train, requires_grad=False)
    y_train_data = nn.Tensor(digit_Y_train, requires_grad=False)
    # y_train_data_one_hot_encoded = nn_core.Tensor(nn_helpers.one_hot(y_train_data.data).T, requires_grad=False)
    y_train_data_one_hot_encoded = nn.Tensor(nn.one_hot(y_train_data.data).T, requires_grad=False)

    # data tracking
    epoch_loss = 0.0
    epoch_accuracy = 0.0

    # training process
    module_test_optimizer.zero_grad()
    logits = module_test_model(x_train_data)

    print ("forward pass done")
    # print (logits.data.shape)

    loss_node = nn.SoftmaxCrossEntropyLossOp(logits, y_train_data_one_hot_encoded)
    loss = loss_node.forward()

    loss.backward()

    module_test_optimizer.step()

    print ("backwardpass done")
    

    epoch_accuracy = nn.compute_accuracy(logits.data, y_train_data.data)
    epoch_loss = loss.data

    print(f"Epoch {epoch} Results -> Loss: {epoch_loss} | Accuracy: {epoch_accuracy}%")
    print()

forward pass done
backwardpass done
Epoch 1 Results -> Loss: 3.019484066871913 | Accuracy: 0.06866071428571428%

forward pass done
backwardpass done
Epoch 2 Results -> Loss: 1.861644593643403 | Accuracy: 0.3836607142857143%

forward pass done
backwardpass done
Epoch 3 Results -> Loss: 1.7489017491572216 | Accuracy: 0.48327380952380955%

forward pass done
backwardpass done
Epoch 4 Results -> Loss: 1.7750378641015232 | Accuracy: 0.609375%

forward pass done
backwardpass done
Epoch 5 Results -> Loss: 1.087964478570361 | Accuracy: 0.6851190476190476%

forward pass done
backwardpass done
Epoch 6 Results -> Loss: 0.7745256413822785 | Accuracy: 0.7611309523809524%

forward pass done
backwardpass done
Epoch 7 Results -> Loss: 0.6603507011365047 | Accuracy: 0.7844642857142857%

forward pass done
backwardpass done
Epoch 8 Results -> Loss: 0.5745092344876821 | Accuracy: 0.8085714285714286%

forward pass done
backwardpass done
Epoch 9 Results -> Loss: 0.5099778498390617 | Accuracy: 0.8441071428571

In [29]:
x_test_data = nn.Tensor(digit_X_test.T, requires_grad=False)

output_logits = module_test_model(x_test_data)
shifted_output_logits = output_logits.data - np.max(output_logits.data, axis=0, keepdims=True) # to keep stable and prevent overflow
exponents = np.exp(shifted_output_logits)
probabilities = (exponents / np.sum(exponents, axis=0, keepdims=True)).T
print (probabilities.shape)

predictions = np.argmax(probabilities, axis=1)
print (predictions.shape)

(28000, 10)
(28000,)


In [32]:
# np.save(f"/Users/ronitmathur/Desktop/ML/Feed-Forward_NN/Kaggle/Results/revamped_test", predictions)
# ---Kaggle Submission---

# flatten and ensure predictions are integers (0 instead of 0.0)
final_predictions = np.asarray(predictions).flatten().astype(int)

# create df and shift the index to start at 1 (kaggle required format)
df = pd.DataFrame({'Label': final_predictions})
df.index = df.index + 1

# save to a CSV
df.to_csv(f"/Users/ronitmathur/Desktop/ML/Feed-Forward_NN/Kaggle/Results/revamped_test.csv", index_label='ImageId')